In [12]:
using LowLevelFEM, LinearAlgebra, IterativeSolvers

openGeometry("I-beam.geo")
#openGeometry("I-szelveny-transfinite.geo")
openPreProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()


In [13]:
mat = Material("beam")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u)
ndofs(U)

127767

In [14]:
e1 = VectorField(U, "left", [-1, 0, 0])
e2 = VectorField(U, "left", [0, -1, 0])
cs = CoordinateSystem(e1, e2, handedness=:left)

CoordinateSystem(VectorField([[-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;]  …  [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;]], Matrix{Float64}(undef, 0, 0), [0.0], [1850, 1851, 1852, 1853, 1854, 1855, 1856, 1857, 1858, 1859  …  2188, 2189, 2190, 2191, 2192, 2193, 2194, 2195, 2196, 2197], 1, :v3D, Problem("I-beam", :VectorField, 3, 3, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 42589, LowLeve

In [15]:
@time K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U), Ω="beam");
#@time K = Q' * K * Q

  2.082606 seconds (1.41 M allocations: 461.936 MiB, 8.19% gc time, 46.31% compilation time: 100% of which was recompilation)


In [16]:
fx = ScalarField(U, "right", (x,y,z)->-y/100000)
fy = ScalarField(U, "right", (x,y,z)->(x-0)/100000);

In [17]:
f = ∫(U ⋅ [fx, fy, 0], Γ="right");

In [18]:
showDoFResults(f, name="f");

In [19]:
bc = BoundaryCondition("right", ux=fx, uy=fy, uz=0)
bc2 = BoundaryCondition("left", uz=0)
bcr = BoundaryCondition("remote", ux=0, uy=0, uz=0)

BoundaryCondition("remote", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :uz => 0, :ux => 0))

In [20]:
per = MPC(master="right", slave="left");

In [ ]:
using IterativeSolvers, Preconditioners
@time u = solveField(Symmetric(K), f, support=[bcr], coordSys=[cs], mpc=[per]);
#@time u = Q * u;

 15.830424 seconds (2.67 M allocations: 3.809 GiB, 5.09% gc time)


In [23]:
showDoFResults(u, :uvec, name="displacement", factor=1);

In [24]:
u1 = nodesToElements(u, onPhysicalGroup="mid")
ε = (u1 ∘ ∇ + ∇ ∘ u1) / 2

E = mat.E
ν = mat.μ
I = unitTensor(ε)

σ = E / (1+ν) * (ε + ν / (1 - 2ν) * trace(ε) * I);

In [25]:
R = nodePositionVector(U)
R = nodesToElements(R, onPhysicalGroup="mid")

n2 = normalVector(U, "mid")

σ2 = elementsToElements(σ, onPhysicalGroup="mid")

τ2 = σ2 * n2;

In [ ]:
showElementResults(τ2, name="shear stress");

In [ ]:
showDoFResults(n2, name="normal vector");

In [26]:
openPostProcessor()